In [2]:
pip install mlx-vlm

  Using cached mlx_vlm-0.5.0-py3-none-any.whl.metadata (44 kB)
  Using cached mlx-0.31.2-cp310-cp310-macosx_26_0_arm64.whl.metadata (5.9 kB)
  Using cached transformers-5.8.1-py3-none-any.whl.metadata (33 kB)
  Using cached miniaudio-1.71-cp310-cp310-macosx_11_0_arm64.whl.metadata (25 kB)
  Using cached llguidance-1.7.5-cp39-abi3-macosx_11_0_arm64.whl.metadata (10 kB)
  Using cached mlx_lm-0.31.3-py3-none-any.whl.metadata (9.5 kB)
  Using cached mlx_audio-0.4.3-py3-none-any.whl.metadata (28 kB)
  Using cached huggingface_hub-1.14.0-py3-none-any.whl.metadata (14 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached hf_xet-1.5.0-cp37-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached mlx_metal-0.31.2-py3-none-macosx_26_0_arm64.whl.metadata (5.1 kB)
  Using cac

In [6]:
import requests
import base64

# 이미지를 Base64 문자열로 변환하는 함수
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

image_path = "../Data/VizWiz_train/VizWiz_train_00000000.jpg" # 실제 이미지 경로로 변경하세요
image_base64 = encode_image(image_path)

# 백그라운드에서 돌고 있는 Ollama 서버로 요청 보내기
url = 'http://localhost:11434/api/generate'
payload = {
    "model": "llava",
    "prompt": "이 사진에 무엇이 있는지 상세히 묘사해줘.",
    "images": [image_base64],
    "stream": False
}

print("Ollama에 추론 요청 중... (시간이 조금 걸릴 수 있습니다)")
response = requests.post(url, json=payload)
result_data = response.json()

# 안전하게 결과를 확인하는 코드
if 'error' in result_data:
    print("❌ Ollama 에러 발생:", result_data['error'])
    print("해결 힌트: 터미널에서 'ollama pull llava'를 실행해 모델을 다운받았는지 확인하세요.")
else:
    print("✅ 결과:", result_data['response'])

Ollama에 추론 요청 중... (시간이 조금 걸릴 수 있습니다)
✅ 결과:  This image shows a jar of Basil Leaves on a table or countertop. The label on the jar indicates that it contains "Basil" and specifies the contents as "BASIL LEAVES." There is also a nutritional information label at the bottom, which states "NET WT. 0.6 OZ (17 G)." This suggests that the jar weighs approximately 17 grams, or about 0.6 ounces, of basil leaves. The label has a green and white color scheme with a photograph of fresh basil leaves on it, indicating the contents of the jar. The overall context of the image implies that this could be part of a spice collection or kitchen storage. 


In [7]:
import json
import requests
import base64
import os

# --- 1. 기본 경로 설정 ---
# 현재 Notebooks 폴더 기준 상대 경로
ANNOTATION_PATH = "../Data/annotations/val.json" 
IMAGE_DIR = "../Data/VizWiz_val/" # 테스트를 위해 val 폴더 사용

# 이미지를 Base64로 변환하는 함수
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# --- 2. JSON 정답지(Annotation) 불러오기 ---
print("JSON 정답지를 불러오는 중...")
try:
    with open(ANNOTATION_PATH, 'r', encoding='utf-8') as f:
        val_data = json.load(f)
    print(f"총 {len(val_data)}개의 데이터가 확인되었습니다.\n")
except FileNotFoundError:
    print("❌ 오류: val.json 파일을 Data/annotations/ 폴더에 넣어주세요!")
    val_data = []

# --- 3. 자동화 반복문 (최초 5장만 테스트) ---
if val_data:
    url = 'http://localhost:11434/api/generate'
    
    # 전체를 다 돌리면 오래 걸리니, [:5]를 써서 딱 5장만 테스트합니다.
    for i, item in enumerate(val_data[:5]):
        image_filename = item['image']      # 예: 'VizWiz_val_00000000.jpg'
        question = item['question']         # 시각장애인이 던진 실제 질문
        
        image_path = os.path.join(IMAGE_DIR, image_filename)
        
        print(f"--- [테스트 {i+1}/5] ---")
        print(f"📷 이미지: {image_filename}")
        print(f"❓ 질문: {question}")
        
        # 파일이 실제로 존재하는지 확인
        if not os.path.exists(image_path):
            print("❌ 이미지 파일이 없습니다. 경로를 확인해주세요.\n")
            continue
            
        # 프롬프트 생성 (실제 질문을 모델에게 던짐)
        prompt = f"You are a helpful assistant for a visually impaired person. Please answer the question based on the image.\nQuestion: {question}"
        
        payload = {
            "model": "llava",
            "prompt": prompt,
            "images": [encode_image(image_path)],
            "stream": False
        }
        
        # Ollama에 추론 요청
        try:
            response = requests.post(url, json=payload)
            result = response.json().get('response', '에러: 답변을 받지 못함')
            print(f"🤖 LLaVA 답변: {result}\n")
        except Exception as e:
            print(f"❌ 요청 중 에러 발생: {e}\n")

print("✅ 샘플 테스트가 완료되었습니다!")

JSON 정답지를 불러오는 중...
총 4319개의 데이터가 확인되었습니다.

--- [테스트 1/5] ---
📷 이미지: VizWiz_val_00000000.jpg
❓ 질문: Ok. There is another picture I hope it is a better one.
🤖 LLaVA 답변:  It seems like there has been a misunderstanding. The image you've provided appears to be of a computer screen displaying an error message. The text in the error message reads, "Computer automatically repair this". This typically indicates that Windows Defender SmartScreen has detected a potentially harmful application or file and is attempting to resolve the issue by removing the file or application from your system.

If you are experiencing issues with your computer due to this error, here's what you can do:

1. Check if there is any update pending for Windows Defender SmartScreen. If not, ensure that all your security updates are installed.

2. Restart your computer in Safe Mode with Networking. In Safe Mode, Windows Defender SmartScreen will be disabled, and you can attempt to access the internet or perform the action

In [10]:
import time

import json
import requests
import base64
import os

# --- 1. 기본 경로 설정 ---
# 현재 Notebooks 폴더 기준 상대 경로
ANNOTATION_PATH = "../Data/annotations/val.json" 
IMAGE_DIR = "../Data/VizWiz_val/" # 테스트를 위해 val 폴더 사용

# 이미지를 Base64로 변환하는 함수
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# --- 2. JSON 정답지(Annotation) 불러오기 ---
print("JSON 정답지를 불러오는 중...")
try:
    with open(ANNOTATION_PATH, 'r', encoding='utf-8') as f:
        val_data = json.load(f)
    print(f"총 {len(val_data)}개의 데이터가 확인되었습니다.\n")
except FileNotFoundError:
    print("❌ 오류: val.json 파일을 Data/annotations/ 폴더에 넣어주세요!")
    val_data = []

# --- 3. 자동화 반복문 ---
# 결과를 저장할 리스트
results_list = []
# 저장할 파일 경로 (Results 폴더 안에 저장)
OUTPUT_FILE = "../Results/baseline_inference_results.json"

if val_data:
    url = 'http://localhost:11434/api/generate'
    
    print("🚀 본격적인 추론을 시작합니다. (저장 기능 포함)")
    # 우선 5장만 테스트로 저장해보고, 잘 되면 [:5]를 지우고 전체를 돌리시면 됩니다.
    for i, item in enumerate(val_data): 
        image_filename = item['image']
        question = item['question']
        image_path = os.path.join(IMAGE_DIR, image_filename)
        
        if not os.path.exists(image_path):
            continue
            
        prompt = f"You are a helpful assistant for a visually impaired person. Please answer the question based on the image.\nQuestion: {question}"
        payload = {
            "model": "llava",
            "prompt": prompt,
            "images": [encode_image(image_path)],
            "stream": False
        }
        
        try:
            response = requests.post(url, json=payload)
            result = response.json().get('response', '에러: 답변을 받지 못함')
            print(f"[{i+1}] {image_filename} 처리 완료")
            
            # 결과를 딕셔너리로 만들어서 리스트에 추가
            results_list.append({
                "image": image_filename,
                "question": question,
                "answer": result
            })
        except Exception as e:
            print(f"[{i+1}] {image_filename} 에러 발생: {e}")
            
    # 반복문이 끝나면 리스트를 JSON 파일로 저장
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(results_list, f, ensure_ascii=False, indent=4) # <-- load를 dump로 변경
        
    print(f"\n✅ 테스트 완료! 결과가 {OUTPUT_FILE}에 안전하게 저장되었습니다.")

JSON 정답지를 불러오는 중...
총 4319개의 데이터가 확인되었습니다.

🚀 본격적인 추론을 시작합니다. (저장 기능 포함)
[1] VizWiz_val_00000000.jpg 처리 완료
[2] VizWiz_val_00000001.jpg 처리 완료
[3] VizWiz_val_00000002.jpg 처리 완료
[4] VizWiz_val_00000003.jpg 처리 완료
[5] VizWiz_val_00000004.jpg 처리 완료
[6] VizWiz_val_00000005.jpg 처리 완료
[7] VizWiz_val_00000006.jpg 처리 완료
[8] VizWiz_val_00000007.jpg 처리 완료
[9] VizWiz_val_00000008.jpg 처리 완료
[10] VizWiz_val_00000009.jpg 처리 완료
[11] VizWiz_val_00000010.jpg 처리 완료
[12] VizWiz_val_00000011.jpg 처리 완료
[13] VizWiz_val_00000012.jpg 처리 완료
[14] VizWiz_val_00000013.jpg 처리 완료
[15] VizWiz_val_00000014.jpg 처리 완료
[16] VizWiz_val_00000015.jpg 처리 완료
[17] VizWiz_val_00000016.jpg 처리 완료
[18] VizWiz_val_00000017.jpg 처리 완료
[19] VizWiz_val_00000018.jpg 처리 완료
[20] VizWiz_val_00000019.jpg 처리 완료
[21] VizWiz_val_00000020.jpg 처리 완료
[22] VizWiz_val_00000021.jpg 처리 완료
[23] VizWiz_val_00000022.jpg 처리 완료
[24] VizWiz_val_00000023.jpg 처리 완료
[25] VizWiz_val_00000024.jpg 처리 완료
[26] VizWiz_val_00000025.jpg 처리 완료
[27] VizWiz_val_00000026.j